# Project 4 · Listening Past the Noise — Step 1: HPSS-Dual Fisher + 8-View MKL + Debiased-Likelihood EM

**Architecture Summary (100% Classical ML / Bishop PRML — strictly compliant with Problem Statement 4 rules):**
1. **Harmonic-Percussive Source Separation (HPSS) Representations:**
   - `HPSS-178d`: Compact active-region (`>-20 dB`) + top-20% energy harmonic/percussive spectral stats, MFCCs, centroids, flatness, and energy ratios.
   - `HPSS-Dual 24+24m Patches (w=8, 32)` → **2-Seed Bagged Fisher Vectors (`PCA=32, GMM=40`)**, **VLAD (`K=32`)**, and **BoAW (`K=96`)**.
2. **Complemented by Standard Views:**
   - `Rich-319d` recording stats, `Mod_v1-468d` (10ms-hop acoustic modulation & cross-band correlation), and Standard `40-mel` Patch `BoAW / VLAD / FV (PCA=24, GMM=32)`.
3. **7-Stream Diverse Classical Ensemble (trained with prior-debiased 6-view Stage-1 teacher, verified `w=1.0`, top-1 agree `w=0.50`, floor `w=0.08`):**
   - `mkl` (`28%`): 8-View Joint Multiple Kernel Learning Precomputed SVM
   - `jlr` (`18%`): L2 Multinomial Logistic Regression on concatenated unit-norm `[FV_H1, FV_H2, FV_Std, VLAD_H, VLAD_Std]`
   - `hpss_k` (`15%`): Dedicated HPSS-Dual Kernel SVM (`0.60 FV_H + 0.40 VLAD_H`)
   - `std_k` (`10%`): Dedicated Standard Log-Mel Kernel SVM (`0.55 FV_S + 0.30 VLAD_S + 0.15 BoAW_S`)
   - `tab` (`12%`): `[Rich 319d + HPSS 178d]` 5-Model Balanced Ensemble (`SVM3 + SVM10 + LR + RF + ET`)
   - `mod` (`10%`): Acoustic Modulation SVM (`50% keep-only + 50% all-unchecked`)
   - `knn` (`7%`): Class-Balanced Verified `16-NN` in the 8-View MKL Kernel Space (zero-noise, zero-domain-shift anchor)
4. **Exact Debiased-Likelihood EM Calibration:**
   - Divides out empirical uniform-weighted 3-seed OOF class marginals $\hat{\pi}_{\text{eff}}(k)$ to recover unbiased likelihoods $L_k(x) \propto p(x \mid y=k)$, fits temperature $T$ on uniform-weighted OOF NLL, and applies damped Saerens EM (`power=0.50`) to adapt cleanly to test class priors.
5. **Validated 3-Seed 5-Fold CV (`SEEDS=[42, 913, 2026]`):** `plain = 0.8722 | unif = 0.8881 | w_tot = 0.9015` (vs. `Highest_Score.ipynb` `0.8374 / 0.8397 / 0.8592`).


In [ ]:
import os, gc, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.io import wavfile
from scipy.signal import periodogram
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel

warnings.filterwarnings("ignore")
T_GLOBAL = time.time()

# Auto-detect Kaggle or local project4 directory
if Path("/kaggle/input").exists() and list(Path("/kaggle/input").rglob("train.csv")):
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "/kaggle/input"))
    WORK = Path(os.environ.get("P4_WORK", "/kaggle/working"))
else:
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "project4"))
    WORK = Path(os.environ.get("P4_WORK", "work_step1"))
WORK.mkdir(parents=True, exist_ok=True)
N_JOBS = int(os.environ.get("P4_JOBS", -1))
SEEDS = [42, 913, 2026]

train_csv = list(INPUT_ROOT.rglob("train.csv"))
assert len(train_csv) == 1, f"Expected 1 train.csv under {INPUT_ROOT}, found {train_csv}"
DATA = train_csv[0].parent
train_df  = pd.read_csv(DATA / "train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA / "test.csv", dtype={"id": str})
sample_df = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})
TRAIN_AUDIO = list(INPUT_ROOT.rglob("audio/train"))[0]
TEST_AUDIO  = list(INPUT_ROOT.rglob("audio/test"))[0]

assert train_df["id"].is_unique and test_df["id"].is_unique and sample_df["id"].is_unique
assert set(test_df["id"]) == set(sample_df["id"]), "test.csv and sample_submission.csv IDs differ"

classes = np.array(sorted(train_df["label"].unique()))
K = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}
verified_mask = (train_df["checked"].values == 1)
y_all_idx = train_df["label"].map(cls_idx).values
yv = y_all_idx[verified_mask]
yu = y_all_idx[~verified_mask]

n_tr, n_te = len(train_df), len(test_df)
ver_ids  = np.flatnonzero(verified_mask)
unc_ids  = np.flatnonzero(~verified_mask)
test_ids = np.arange(n_tr, n_tr + n_te)

ver_cnt = np.bincount(yv, minlength=K)
tot_cnt = np.bincount(y_all_idx, minlength=K)
w_unif  = (1.0 / ver_cnt)[yv]
w_tot   = ((tot_cnt / tot_cnt.sum()) / (ver_cnt / ver_cnt.sum()))[yv]
v40     = (ver_cnt == ver_cnt.max())

print(f"Data dir: {DATA} | Work dir: {WORK}")
print(f"Train: {n_tr} (verified={len(yv)}, unchecked={len(yu)}) | Test: {n_te} | Classes: {K}")

## 1. Consolidated Single-Pass Audio Feature & Patch Extraction
Each WAV clip is read once to extract all five Step 1 raw representations:
- `Rich-319d` recording stats (exact benchmark specification)
- `HPSS-178d` active (`>-20 dB`) + top-20% energy harmonic/percussive recording stats
- `Mod_v1-468d` (10ms-hop 24-mel envelope cross-band correlations & 0.5–50 Hz modulation spectra)
- `Std 40-mel` short-window patches (`w = 8, 32`)
- `HPSS-Dual 24+24m` short-window patches (`w = 8, 32`)


In [ ]:
SR, N_FFT, HOP = 22050, 2048, 512
MEL32_2048 = librosa.filters.mel(sr=SR, n_fft=2048, n_mels=32).astype(np.float32)
MEL24_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=24).astype(np.float32)
MEL40_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=40).astype(np.float32)

def count_onsets(env, sr, hop):
    if not np.isfinite(env).all() or not np.any(env):
        return 0
    normalized = env - env.min()
    normalized = normalized / (normalized.max() + np.finfo(env.dtype).tiny)
    pre_max = int(0.03 * sr // hop)
    pre_avg = int(0.10 * sr // hop)
    post_avg = pre_avg + 1
    wait = pre_max
    count = 0
    previous = -wait - 1
    for i, value in enumerate(normalized):
        if i <= previous + wait:
            continue
        peak_window = normalized[max(0, i - pre_max):i + 1]
        avg_window = normalized[max(0, i - pre_avg):i + post_avg]
        if value == peak_window.max() and value >= avg_window.mean() + 0.07:
            count += 1
            previous = i
    return count

def mask_slice(M, mask):
    n = min(M.shape[1], len(mask))
    m = mask[:n]
    if m.sum() < 2:
        m = np.ones(n, bool)
    return M[:, :n][:, m]

def make_window_patches(mat, widths=(8, 32)):
    views = []
    for w in widths:
        h = w // 2
        fr = np.pad(mat, ((0, max(0, w - len(mat))), (0, 0)), mode="edge")
        starts = list(range(0, len(fr) - w + 1, h))
        if starts[-1] != len(fr) - w:
            starts.append(len(fr) - w)
        bl = np.stack([fr[i:i + w] for i in starts])
        hf = w // 2
        views.append(np.hstack([bl.mean(1), bl.std(1), bl[:, hf:].mean(1) - bl[:, :hf].mean(1)]).astype(np.float32))
    return views

def extract_all_views_one(path):
    rate, pcm = wavfile.read(str(path))
    if pcm.ndim > 1:
        pcm = pcm.mean(axis=1).astype(np.int16)
    y = pcm.astype(np.float32) / 32768.0
    sr = rate
    dur = len(y) / sr
    y_pad = np.pad(y, (0, N_FFT - len(y))) if len(y) < N_FFT else y

    # ── 1. Rich-319d (exact benchmark) ──
    S = np.abs(librosa.stft(y_pad, n_fft=N_FFT, hop_length=HOP))
    P2 = S ** 2
    rms  = librosa.feature.rms(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    zcr  = librosa.feature.zero_crossing_rate(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
    chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
    cent = librosa.feature.spectral_centroid(S=S, sr=sr)
    bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
    roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
    con  = librosa.feature.spectral_contrast(S=S, sr=sr)
    ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
    base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
    contrast = ms(con)
    r1 = rms[0]; T = S.shape[1]
    act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
    if act.sum() < 3:
        act = np.ones(T, bool)
    lm = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
    lma, mfa = lm[:, act], mfcc[:, act]
    dm = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
    flat = librosa.feature.spectral_flatness(S=S)[0]
    env = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
    n_on = count_onsets(env, sr, HOP)
    lr = np.log(r1 + 1e-8)
    rich = np.concatenate([
        lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
        mfa.mean(1), mfa.std(1), dm.std(1),
        [flat.mean(), flat.std(), env.mean(), env.std()],
        np.percentile(lr, [10, 50, 90]),
        [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]
    ])
    f_rich = np.concatenate([base, contrast, rich]).astype(np.float32)

    # ── 2. HPSS-178d (reuses 2048-FFT STFT scaled by peak!) ──
    raw_rms = float(np.sqrt(np.mean(y * y)) + 1e-8)
    raw_peak = float(np.max(np.abs(y)) + 1e-8)
    S_n = S / raw_peak
    P_n = S_n ** 2
    S_h, S_p = librosa.decompose.hpss(S_n, margin=2.0)
    P_h, P_p = S_h ** 2, S_p ** 2
    fe = P_n.sum(axis=0)
    fe_db = 10.0 * np.log10(fe + 1e-10)
    act20 = fe_db >= (fe_db.max() - 20.0)
    if act20.sum() < 4: act20 = fe >= np.percentile(fe, 70)
    if act20.sum() < 2: act20 = np.ones(len(fe), bool)
    top20 = fe >= np.percentile(fe, 80)
    if top20.sum() < 2: top20 = np.ones(len(fe), bool)
    h_e = P_h.sum(0); p_e = P_p.sum(0); t_e = fe + 1e-10
    h_rat = h_e / t_e; p_rat = p_e / t_e
    mel_h32 = librosa.power_to_db(MEL32_2048 @ P_h, ref=np.max)
    mel_p32 = librosa.power_to_db(MEL32_2048 @ P_p, ref=np.max)
    mfcc_h = librosa.feature.mfcc(S=mel_h32, n_mfcc=16)
    mfcc_p = librosa.feature.mfcc(S=mel_p32, n_mfcc=16)
    cent_h = librosa.feature.spectral_centroid(S=S_h, sr=sr)[0]
    cent_p = librosa.feature.spectral_centroid(S=S_p, sr=sr)[0]
    flat_h = librosa.feature.spectral_flatness(S=S_h + 1e-8)[0]
    flat_p = librosa.feature.spectral_flatness(S=S_p + 1e-8)[0]
    def stats3(M, mask):
        A = mask_slice(M, mask)
        return np.concatenate([A.mean(1), A.std(1), np.percentile(A, 90, axis=1)])
    f_hpss = np.concatenate([
        [h_rat.mean(), h_rat.std(), h_rat[act20].mean(), p_rat.mean(), p_rat.std(), p_rat[act20].mean(),
         (h_e.sum() / (t_e.sum() + 1e-10)), (p_e.sum() / (t_e.sum() + 1e-10)),
         cent_h[act20].mean(), cent_h[act20].std(), cent_p[act20].mean(), cent_p[act20].std(),
         flat_h[act20].mean(), flat_p[act20].mean(),
         np.log(raw_rms), np.log(raw_peak), raw_peak / raw_rms, act20.mean()],
        stats3(mfcc_h, act20), stats3(mfcc_p, act20),
        mask_slice(mel_h32, top20).mean(1), mask_slice(mel_p32, top20).mean(1),
    ]).astype(np.float32)

    # ── 3. Mod_v1-468d (10ms hop, exact benchmark) ──
    width_m = 1024
    hop_m = max(1, round(sr * 0.010))
    yp_m = np.pad(y, (width_m // 2, width_m // 2))
    frames_m = np.lib.stride_tricks.sliding_window_view(yp_m, width_m)[::hop_m]
    spec_m = np.abs(np.fft.rfft(frames_m * np.hanning(width_m), axis=1)) ** 2
    hz = np.fft.rfftfreq(width_m, 1.0 / sr)
    mel_edges = np.linspace(0, np.log1p(min(11000, sr / 2) / 700), 26)
    edges = 700 * np.expm1(mel_edges)
    bank = np.maximum(0, np.minimum(
        (hz[:, None] - edges[:-2]) / np.diff(edges)[:-1],
        (edges[2:] - hz[:, None]) / np.diff(edges)[1:]))
    energy = spec_m @ bank
    envelope = np.log1p(100 * energy / max(energy.mean(), 1e-10))
    centered = envelope - envelope.mean(axis=0)
    scales = np.sqrt(np.mean(centered ** 2, axis=0))
    norm = centered / np.maximum(scales, 1e-6)
    corr = norm.T @ norm / len(norm)
    freq, power = periodogram(centered, fs=sr / hop_m, axis=0, nfft=max(256, len(centered)), detrend=False, scaling="density")
    bands = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 50)]
    modulation = [np.log1p(power[(freq >= lo) & (freq < hi)].sum(0) * freq[1]) for lo, hi in bands]
    f_mod = np.concatenate([envelope.mean(0), scales, corr[np.triu_indices(24, k=1)], *modulation]).astype(np.float32)

    # ── 4 & 5. Std 40-mel Patches + HPSS-Dual 24+24m Patches (12ms hop, 1024-FFT) ──
    hop_p = max(1, round(sr * 0.012))
    S1 = np.abs(librosa.stft(y, n_fft=1024, hop_length=hop_p))
    P1 = S1 ** 2
    lm40 = librosa.power_to_db(MEL40_1024 @ P1, ref=np.max).T
    p_std = make_window_patches(lm40, (8, 32))

    S1_h, S1_p = librosa.decompose.hpss(S1, margin=2.0)
    lm_h = librosa.power_to_db(MEL24_1024 @ (S1_h ** 2), ref=np.max).T
    lm_p = librosa.power_to_db(MEL24_1024 @ (S1_p ** 2), ref=np.max).T
    p_hpss = make_window_patches(np.hstack([lm_h, lm_p]), (8, 32))

    return f_rich, f_hpss, f_mod, p_std, p_hpss

CACHE_TAB = WORK / "step1_tab_mod_v1.npz"
CACHE_ENC = WORK / "step1_encoded_views_v1.npz"

if CACHE_TAB.exists() and CACHE_ENC.exists():
    print("Loading cached Step 1 features & encoded patch views...", flush=True)
    zt = np.load(CACHE_TAB)
    X_rich_all, X_hpss_all, X_mod_all = zt["X_rich"], zt["X_hpss"], zt["X_mod"]
    ze = np.load(CACHE_ENC)
    boaw_s, vlad_s, fv_s = ze["boaw_s"], ze["vlad_s"], ze["fv_s"]
    boaw_h, vlad_h, fv_h1, fv_h2 = ze["boaw_h"], ze["vlad_h"], ze["fv_h1"], ze["fv_h2"]
else:
    t_ext = time.time()
    all_paths = [TRAIN_AUDIO / f"{i}.wav" for i in train_df["id"]] + [TEST_AUDIO / f"{i}.wav" for i in test_df["id"]]
    print(f"Extracting all 5 views for {len(all_paths)} clips...", flush=True)
    res = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(delayed(extract_all_views_one)(p) for p in all_paths)
    print(f"Raw extraction finished in {time.time() - t_ext:.1f}s", flush=True)

    X_rich_all = np.vstack([r[0] for r in res]).astype(np.float32)
    X_hpss_all = np.vstack([r[1] for r in res]).astype(np.float32)
    X_mod_all  = np.vstack([r[2] for r in res]).astype(np.float32)
    std_patches  = [r[3] for r in res]
    hpss_patches = [r[4] for r in res]
    del res; gc.collect()

    # Clean NaNs strictly using training medians
    X_rich_all = np.where(np.isfinite(X_rich_all), X_rich_all, np.nan)
    med_r = np.nanmedian(X_rich_all[:n_tr], axis=0)
    X_rich_all = np.where(np.isnan(X_rich_all), med_r, X_rich_all).astype(np.float32)
    X_hpss_all = np.nan_to_num(X_hpss_all).astype(np.float32)
    X_mod_all  = np.nan_to_num(X_mod_all).astype(np.float32)
    np.savez_compressed(CACHE_TAB, X_rich=X_rich_all, X_hpss=X_hpss_all, X_mod=X_mod_all)

    # ── Unsupervised Patch Encoding (strictly fit on unchecked training clips!) ──
    def encode_patch_family(patch_list, pca_dim, n_gmm, gmm_seeds, max_per_clip=24, base_seed=42):
        boaw_v, vlad_v = [], []
        fv_v_by_seed = {s: [] for s in gmm_seeds}
        for view in range(2):
            rng = np.random.default_rng(base_seed + view)
            pool = np.vstack([
                patch_list[i][view][rng.choice(len(patch_list[i][view]), min(max_per_clip, len(patch_list[i][view])), replace=False)]
                for i in unc_ids
            ])
            scaler = StandardScaler().fit(pool)
            pool_s = scaler.transform(pool).astype(np.float32)

            # BoAW (K=96)
            km96 = MiniBatchKMeans(n_clusters=96, batch_size=2048, n_init=3, random_state=42).fit(pool_s)
            c96 = km96.cluster_centers_.astype(np.float32)
            def _boaw(rec):
                fr = scaler.transform(rec[view]).astype(np.float32)
                d = cdist(fr, c96, "sqeuclidean")
                nr = np.argpartition(d, 2, axis=1)[:, :3]
                w = 1.0 / np.maximum(np.take_along_axis(d, nr, axis=1), 1e-8)
                w /= w.sum(axis=1, keepdims=True)
                h = np.bincount(nr.ravel(), weights=w.ravel(), minlength=len(c96))
                return np.sqrt(h / max(h.sum(), 1e-12)).astype(np.float32)
            boaw_v.append(np.vstack([_boaw(r) for r in patch_list]))

            # VLAD (K=32)
            km32 = MiniBatchKMeans(n_clusters=32, batch_size=2048, n_init=3, random_state=17).fit(pool_s)
            c32 = km32.cluster_centers_.astype(np.float32)
            def _vlad(rec):
                fr = scaler.transform(rec[view]).astype(np.float32)
                asgn = cdist(fr, c32, "sqeuclidean").argmin(axis=1)
                sums = np.zeros_like(c32)
                np.add.at(sums, asgn, fr - c32[asgn])
                sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
                v = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
                return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
            vlad_v.append(np.stack([_vlad(r) for r in patch_list]))

            # Fisher Vectors (1 or 2 seeds)
            for gs in gmm_seeds:
                pca = PCA(n_components=pca_dim, whiten=True, svd_solver="randomized", random_state=gs)
                proj = pca.fit_transform(pool_s)
                gmm = GaussianMixture(n_gmm, covariance_type="diag", reg_covar=0.01, max_iter=80, random_state=gs).fit(proj)
                mu, var, wt = gmm.means_, gmm.covariances_, gmm.weights_
                def _fv(rec):
                    x = pca.transform(scaler.transform(rec[view]))
                    q = gmm.predict_proba(x)
                    mass = q.sum(axis=0)[:, None]
                    mom = q.T @ x
                    norm = len(x) * np.sqrt(wt[:, None])
                    f1 = (mom - mass * mu) / (norm * np.sqrt(var))
                    f2 = ((q.T @ (x * x) - 2 * mu * mom + mass * mu**2) / var - mass) / (np.sqrt(2) * norm)
                    v = np.concatenate([f1.ravel(), f2.ravel()])
                    v = np.sign(v) * np.sqrt(np.abs(v))
                    return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
                fv_v_by_seed[gs].append(np.stack([_fv(r) for r in patch_list]))
        return (
            np.hstack(boaw_v).astype(np.float32),
            np.hstack(vlad_v).astype(np.float32),
            [np.hstack(fv_v_by_seed[gs]).astype(np.float32) for gs in gmm_seeds],
        )

    print("Encoding Std 40-mel patches (BoAW, VLAD, FV PCA24/GMM32)...", flush=True)
    boaw_s, vlad_s, [fv_s] = encode_patch_family(std_patches, pca_dim=24, n_gmm=32, gmm_seeds=[81], max_per_clip=20, base_seed=42)
    del std_patches; gc.collect()

    print("Encoding HPSS-Dual 24+24m patches (BoAW, VLAD, 2-seed Bagged FV PCA32/GMM40)...", flush=True)
    boaw_h, vlad_h, [fv_h1, fv_h2] = encode_patch_family(hpss_patches, pca_dim=32, n_gmm=40, gmm_seeds=[81, 137], max_per_clip=24, base_seed=42)
    del hpss_patches; gc.collect()

    np.savez(CACHE_ENC, boaw_s=boaw_s, vlad_s=vlad_s, fv_s=fv_s, boaw_h=boaw_h, vlad_h=vlad_h, fv_h1=fv_h1, fv_h2=fv_h2)

X_tab_all = np.hstack([X_rich_all, X_hpss_all]).astype(np.float32)
print(f"Views ready ({time.time()-T_GLOBAL:.0f}s): Rich={X_rich_all.shape[1]}, HPSS={X_hpss_all.shape[1]}, Mod={X_mod_all.shape[1]}, FV_S={fv_s.shape[1]}, FV_H={fv_h1.shape[1]}")

## 2. Kernel Precomputation & Joint Linear Space Construction
Precompute unit-diagonal linear Gram matrices for `FV_H` (2-seed bagged average), `FV_S`, `VLAD_H`, `VLAD_S`, and RBF Gram matrices for `BoAW`, plus the weighted concatenated unit-norm feature matrix `X_jlr` for Multinomial Logistic Regression.


In [ ]:
K_fv_s   = linear_kernel(fv_s, fv_s).astype(np.float32)
K_vl_s   = linear_kernel(vlad_s, vlad_s).astype(np.float32)
K_fv_h   = (0.5 * linear_kernel(fv_h1, fv_h1) + 0.5 * linear_kernel(fv_h2, fv_h2)).astype(np.float32)
K_vl_h   = linear_kernel(vlad_h, vlad_h).astype(np.float32)
boaw_both = np.hstack([boaw_s, boaw_h]).astype(np.float32)
K_bw_all = rbf_kernel(boaw_both, boaw_both, gamma=1.0 / (boaw_both.shape[1] * boaw_both.var())).astype(np.float32)
K_bw_s   = rbf_kernel(boaw_s, boaw_s, gamma=1.0 / (boaw_s.shape[1] * boaw_s.var())).astype(np.float32)

X_jlr_all = np.hstack([
    np.sqrt(0.28) * fv_h1,
    np.sqrt(0.28) * fv_h2,
    np.sqrt(0.26) * fv_s,
    np.sqrt(0.18) * vlad_h,
    np.sqrt(0.14) * vlad_s,
]).astype(np.float32)

del fv_h1, fv_h2, fv_s, vlad_h, vlad_s, boaw_s, boaw_h, boaw_both
gc.collect()
print(f"Gram matrices & X_jlr {X_jlr_all.shape} ready.")

## 3. Leakage-Free 3-Seed 5-Fold Stratified Cross-Validation
Every fold fits its Stage-1 6-view verified teacher strictly on `tr` verified clips, debiases `Pu` using fold-internal OOF marginals, selects `top-1 agree` unchecked clips (`w=0.50`, floor `w=0.08`), trains all 7 streams strictly inside the fold, and evaluates Cross-Fit OOF Marginal Prior-Debiasing across `plain`, `unif`, `w_tot`, `v10`, and `v40` MAP@3.


In [ ]:
def map3_s(yi, P):
    top3 = np.argsort(-P, axis=1)[:, :3]
    return ((top3 == yi[:, None]) / np.array([1.0, 2.0, 3.0])).sum(1)

def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(1, keepdims=True)

def em_prior(P, iters=200, floor=0.25):
    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :]
        Q /= Q.sum(1, keepdims=True)
        pi = Q.mean(0)
    pi = np.clip(pi, floor / K, None)
    return pi / pi.sum()

def fit_temp(P, yi, w):
    best = (1e9, 1.0)
    for t in np.linspace(0.5, 4.0, 36):
        Q = sharpen(P, t)
        nll = -np.average(np.log(Q[np.arange(len(yi)), yi] + 1e-12), weights=w)
        if nll < best[0]:
            best = (nll, t)
    return best[1]

def fit_predict_tab(Xtr, ytr, wtr, X_list, n_jobs=1):
    models = [
        (make_pipeline(StandardScaler(), SVC(C=3, kernel="rbf", class_weight="balanced", probability=True, random_state=42)), "svc__sample_weight", 0.25),
        (make_pipeline(StandardScaler(), SVC(C=10, kernel="rbf", class_weight="balanced", probability=True, random_state=42)), "svc__sample_weight", 0.15),
        (make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)), "logisticregression__sample_weight", 0.10),
        (RandomForestClassifier(n_estimators=500, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=n_jobs), "sample_weight", 0.25),
        (ExtraTreesClassifier(n_estimators=500, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=n_jobs), "sample_weight", 0.25),
    ]
    outs = [np.zeros((len(X), K), dtype=np.float64) for X in X_list]
    for m, wk, a in models:
        if wtr is None: m.fit(Xtr, ytr)
        else:           m.fit(Xtr, ytr, **{wk: wtr})
        for o, X in zip(outs, X_list):
            o[:, m.classes_] += a * m.predict_proba(X)
    return outs

def fit_stage1_teacher(v_tr_ids, y_tr, target_ids, n_jobs_tab=1):
    p_tab = fit_predict_tab(X_tab_all[v_tr_ids], y_tr, None, [X_tab_all[target_ids]], n_jobs=n_jobs_tab)[0]
    sc_m = StandardScaler().fit(X_mod_all[v_tr_ids])
    p_mod = SVC(C=8, kernel="rbf", class_weight="balanced", probability=True, random_state=42).fit(
        sc_m.transform(X_mod_all[v_tr_ids]), y_tr).predict_proba(sc_m.transform(X_mod_all[target_ids]))
    K_t = 0.42 * K_fv_h + 0.24 * K_fv_s + 0.18 * K_vl_h + 0.10 * K_vl_s + 0.06 * K_bw_all
    p_k = SVC(kernel="precomputed", C=12, class_weight="balanced", probability=True, random_state=42).fit(
        K_t[np.ix_(v_tr_ids, v_tr_ids)], y_tr).predict_proba(K_t[np.ix_(target_ids, v_tr_ids)])
    p_lr = LogisticRegression(C=10.0, class_weight="balanced", max_iter=400).fit(
        X_jlr_all[v_tr_ids], y_tr).predict_proba(X_jlr_all[target_ids])
    return 0.28 * p_tab + 0.14 * p_mod + 0.34 * p_k + 0.24 * p_lr

def fit_all_streams(v_tr_ids, y_tr, target_ids, keep, n_jobs_tab=1):
    all_ids = np.r_[v_tr_ids, unc_ids]
    all_y   = np.r_[y_tr, yu]
    all_w   = np.r_[np.ones(len(v_tr_ids)), np.where(keep, 0.50, 0.08)]

    keep_ids = np.r_[v_tr_ids, unc_ids[keep]]
    keep_y   = np.r_[y_tr, yu[keep]]
    keep_w   = np.r_[np.ones(len(v_tr_ids)), np.full(keep.sum(), 0.50)]

    sc_r = StandardScaler().fit(X_rich_all[all_ids])
    sc_h = StandardScaler().fit(X_hpss_all[all_ids])
    sc_m = StandardScaler().fit(X_mod_all[all_ids])
    Xr_tr, Xr_tg = sc_r.transform(X_rich_all[all_ids]), sc_r.transform(X_rich_all[target_ids])
    Xh_tr, Xh_tg = sc_h.transform(X_hpss_all[all_ids]), sc_h.transform(X_hpss_all[target_ids])
    Xm_tr, Xm_tg = sc_m.transform(X_mod_all[all_ids]),  sc_m.transform(X_mod_all[target_ids])

    Kr_tr = rbf_kernel(Xr_tr, Xr_tr, gamma=1.0 / Xr_tr.shape[1]).astype(np.float32)
    Kr_tg = rbf_kernel(Xr_tg, Xr_tr, gamma=1.0 / Xr_tr.shape[1]).astype(np.float32)
    Kh_tr = rbf_kernel(Xh_tr, Xh_tr, gamma=1.0 / Xh_tr.shape[1]).astype(np.float32)
    Kh_tg = rbf_kernel(Xh_tg, Xh_tr, gamma=1.0 / Xh_tr.shape[1]).astype(np.float32)
    Km_tr = rbf_kernel(Xm_tr, Xm_tr, gamma=1.0 / Xm_tr.shape[1]).astype(np.float32)
    Km_tg = rbf_kernel(Xm_tg, Xm_tr, gamma=1.0 / Xm_tr.shape[1]).astype(np.float32)

    # Stream 1: 8-View Joint MKL Precomputed SVM
    K_mkl_tr = (0.24 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.16 * K_fv_s[np.ix_(all_ids, all_ids)] +
                0.14 * K_vl_h[np.ix_(all_ids, all_ids)] + 0.10 * K_vl_s[np.ix_(all_ids, all_ids)] +
                0.06 * K_bw_all[np.ix_(all_ids, all_ids)] + 0.11 * Kr_tr + 0.10 * Kh_tr + 0.09 * Km_tr)
    K_mkl_tg = (0.24 * K_fv_h[np.ix_(target_ids, all_ids)] + 0.16 * K_fv_s[np.ix_(target_ids, all_ids)] +
                0.14 * K_vl_h[np.ix_(target_ids, all_ids)] + 0.10 * K_vl_s[np.ix_(target_ids, all_ids)] +
                0.06 * K_bw_all[np.ix_(target_ids, all_ids)] + 0.11 * Kr_tg + 0.10 * Kh_tg + 0.09 * Km_tg)
    p_mkl = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_mkl_tr, all_y, sample_weight=all_w).predict_proba(K_mkl_tg)

    # Stream 2: Joint L2 Multinomial Logistic Regression
    p_jlr = LogisticRegression(C=12.0, max_iter=500).fit(X_jlr_all[all_ids], all_y, sample_weight=all_w).predict_proba(X_jlr_all[target_ids])

    # Stream 3: HPSS-Dual Kernel SVM
    K_hp_tr = 0.60 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.40 * K_vl_h[np.ix_(all_ids, all_ids)]
    K_hp_tg = 0.60 * K_fv_h[np.ix_(target_ids, all_ids)] + 0.40 * K_vl_h[np.ix_(target_ids, all_ids)]
    p_hpss_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_hp_tr, all_y, sample_weight=all_w).predict_proba(K_hp_tg)

    # Stream 4: Standard Log-Mel Kernel SVM
    K_st_tr = 0.55 * K_fv_s[np.ix_(all_ids, all_ids)] + 0.30 * K_vl_s[np.ix_(all_ids, all_ids)] + 0.15 * K_bw_s[np.ix_(all_ids, all_ids)]
    K_st_tg = 0.55 * K_fv_s[np.ix_(target_ids, all_ids)] + 0.30 * K_vl_s[np.ix_(target_ids, all_ids)] + 0.15 * K_bw_s[np.ix_(target_ids, all_ids)]
    p_std_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_st_tr, all_y, sample_weight=all_w).predict_proba(K_st_tg)

    # Stream 5: Rich + HPSS (497d) 5-Model Balanced Tabular Ensemble
    p_tab = fit_predict_tab(X_tab_all[keep_ids], keep_y, keep_w, [X_tab_all[target_ids]], n_jobs=n_jobs_tab)[0]

    # Stream 6: Modulation v1 (50% keep + 50% all-unchecked)
    sc_mk = StandardScaler().fit(X_mod_all[keep_ids])
    p_mod_k = SVC(C=8, kernel="rbf", probability=True, random_state=42).fit(
        sc_mk.transform(X_mod_all[keep_ids]), keep_y, sample_weight=keep_w).predict_proba(sc_mk.transform(X_mod_all[target_ids]))
    p_mod_a = SVC(kernel="precomputed", C=10, probability=True, random_state=42).fit(Km_tr, all_y, sample_weight=all_w).predict_proba(Km_tg)
    p_mod = 0.50 * p_mod_k + 0.50 * p_mod_a

    # Stream 7: Class-Balanced Verified 16-NN in 8-View MKL Kernel Space
    n_vtr = len(v_tr_ids)
    S_v = K_mkl_tg[:, :n_vtr]
    cnt_tr = np.bincount(y_tr, minlength=K)
    inv_c = (1.0 / np.maximum(cnt_tr, 1))[y_tr]
    p_knn = np.zeros((len(target_ids), K), dtype=np.float64)
    k_nn, tau = 16, 15.0
    for i in range(len(target_ids)):
        sims = S_v[i]
        top = np.argpartition(-sims, k_nn)[:k_nn]
        w_n = np.exp(tau * (sims[top] - sims[top].max())) * inv_c[top]
        votes = np.bincount(y_tr[top], weights=w_n, minlength=K)
        p_knn[i] = votes / np.maximum(votes.sum(), 1e-12)

    return dict(mkl=p_mkl, jlr=p_jlr, hpss_k=p_hpss_k, std_k=p_std_k, tab=p_tab, mod=p_mod, knn=p_knn)

W_FIXED = dict(mkl=0.28, jlr=0.18, hpss_k=0.15, std_k=0.10, tab=0.12, mod=0.10, knn=0.07)
oofs_s1 = {}
blend_raw_by_seed = {}
splits_by_seed = {}

for seed in SEEDS:
    ts = time.time()
    splits = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(ver_ids, yv))
    splits_by_seed[seed] = splits
    s1_res = Parallel(n_jobs=N_JOBS)(
        delayed(fit_stage1_teacher)(ver_ids[tr_i], yv[tr_i], np.r_[ver_ids[va_i], unc_ids])
        for tr_i, va_i in splits
    )
    oof_t = np.zeros((len(yv), K))
    Pu_list = []
    for (tr_i, va_i), P_out in zip(splits, s1_res):
        oof_t[va_i] = P_out[:len(va_i)]
        Pu_list.append(P_out[len(va_i):])
    oofs_s1[seed] = oof_t

    def run_fold(fi):
        tr_i, va_i = splits[fi]
        Pu = Pu_list[fi]
        wu = w_unif[tr_i] / w_unif[tr_i].sum()
        pi_t = (oof_t[tr_i] * wu[:, None]).sum(0)
        Pu_deb = Pu / np.maximum(pi_t[None, :], 1e-6)
        Pu_deb /= Pu_deb.sum(1, keepdims=True)
        keep = (Pu_deb > Pu_deb[np.arange(len(yu)), yu][:, None]).sum(1) < 1
        d = fit_all_streams(ver_ids[tr_i], yv[tr_i], ver_ids[va_i], keep, n_jobs_tab=1)
        return sum(W_FIXED[nm] * d[nm] for nm in W_FIXED)

    fold_preds = Parallel(n_jobs=N_JOBS)(delayed(run_fold)(fi) for fi in range(5))
    oof_b = np.zeros((len(yv), K))
    for (_, va_i), P_va in zip(splits, fold_preds):
        oof_b[va_i] = P_va
    blend_raw_by_seed[seed] = oof_b

    # Cross-fit OOF marginal debiasing
    oof_deb = np.zeros_like(oof_b)
    for tr_i, va_i in splits:
        wu = w_unif[tr_i] / w_unif[tr_i].sum()
        pi_m = (oof_b[tr_i] * wu[:, None]).sum(0)
        Q = oof_b[va_i] / np.maximum(pi_m[None, :], 1e-6)
        oof_deb[va_i] = Q / Q.sum(1, keepdims=True)
    sc = map3_s(yv, oof_deb)
    print(f"Seed {seed} ({time.time()-ts:.0f}s) | Debiased plain={sc.mean():.5f} | unif={np.average(sc,weights=w_unif):.5f} | w_tot={np.average(sc,weights=w_tot):.5f} | v10={sc[~v40[yv]].mean():.5f} | v40={sc[v40[yv]].mean():.5f}", flush=True)

# Mean 3-seed CV summary
pl, un, wt, v10s, v40s = [], [], [], [], []
for s in SEEDS:
    P = blend_raw_by_seed[s]
    Q_out = np.zeros_like(P)
    for tr_i, va_i in splits_by_seed[s]:
        wu = w_unif[tr_i] / w_unif[tr_i].sum()
        pi_m = (P[tr_i] * wu[:, None]).sum(0)
        Q = P[va_i] / np.maximum(pi_m[None, :], 1e-6)
        Q_out[va_i] = Q / Q.sum(1, keepdims=True)
    sc = map3_s(yv, Q_out)
    pl.append(sc.mean()); un.append(np.average(sc, weights=w_unif)); wt.append(np.average(sc, weights=w_tot))
    v10s.append(sc[~v40[yv]].mean()); v40s.append(sc[v40[yv]].mean())
print(f"\n3-SEED MEAN CV | plain={np.mean(pl):.5f} | unif={np.mean(un):.5f} | w_tot={np.mean(wt):.5f} | v10={np.mean(v10s):.5f} | v40={np.mean(v40s):.5f}")

## 4. Full-Data Retraining (All 620 Verified + 4,984 Unchecked) & Exact Debiased-Likelihood EM Submission


In [ ]:
t_fin = time.time()
wu_all = w_unif / w_unif.sum()

# 1. Full 620-verified Stage-1 teacher debiased by 3-seed OOF marginal
oof_s1_mean = np.mean([oofs_s1[s] for s in SEEDS], axis=0)
pi_s1 = (oof_s1_mean * wu_all[:, None]).sum(0)
Pu_full = fit_stage1_teacher(ver_ids, yv, unc_ids, n_jobs_tab=N_JOBS)
Pu_full_deb = Pu_full / np.maximum(pi_s1[None, :], 1e-6)
Pu_full_deb /= Pu_full_deb.sum(1, keepdims=True)
keep_full = (Pu_full_deb > Pu_full_deb[np.arange(len(yu)), yu][:, None]).sum(1) < 1
print(f"Full Stage-1 teacher kept {keep_full.sum()}/{len(yu)} unchecked clips ({100*keep_full.mean():.1f}%)")

# 2. Train all 7 streams on 100% of verified + unchecked -> predict test
test_streams = fit_all_streams(ver_ids, yv, test_ids, keep_full, n_jobs_tab=N_JOBS)
P_test_raw = sum(W_FIXED[nm] * test_streams[nm] for nm in W_FIXED)

# 3. Exact Debiased-Likelihood EM Calibration
P_oof_mean = np.mean([blend_raw_by_seed[s] for s in SEEDS], axis=0)
pi_src = (P_oof_mean * wu_all[:, None]).sum(0)
P_oof_deb = P_oof_mean / np.maximum(pi_src[None, :], 1e-6)
P_oof_deb /= P_oof_deb.sum(1, keepdims=True)
temp_opt = fit_temp(P_oof_deb, yv, w_unif)

P_test_deb = P_test_raw / np.maximum(pi_src[None, :], 1e-6)
P_test_deb /= P_test_deb.sum(1, keepdims=True)
P_test_cal = sharpen(P_test_deb, temp_opt)
pi_test_hat = em_prior(P_test_cal)
P_test_final = P_test_cal * (pi_test_hat ** 0.50)[None, :]
P_test_final /= P_test_final.sum(1, keepdims=True)

print(f"Calibration: temp_opt={temp_opt:.2f} | test pi_hat min={100*pi_test_hat.min():.2f}% max={100*pi_test_hat.max():.2f}%")

# 4. Write & strictly verify submission.csv
top3 = np.argsort(-P_test_final, axis=1)[:, :3]
sub = pd.DataFrame({"id": test_df["id"], "label": [" ".join(classes[r]) for r in top3]})
sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
assert sub.shape == (len(sample_df), 2), f"Bad shape: {sub.shape}"
assert sub["label"].notna().all() and sub["id"].is_unique
assert all(len(set(s.split())) == 3 and all(c in cls_idx for c in s.split()) for s in sub["label"])

sub_path = WORK / "submission.csv"
sub.to_csv(sub_path, index=False, lineterminator="\n")
print(f"VERIFIED SUBMISSION SAVED: {sub_path} {sub.shape} | Retrain: {time.time()-t_fin:.0f}s | Total: {(time.time()-T_GLOBAL)/60:.1f} min")
print("Top-1 classes represented:", sub["label"].str.split().str[0].nunique(), "/", K)
print(sub.head(10).to_string(index=False))

## Summary and evaluation

### Implementation flow (Submission 13 — Step 1)
1. **Five-view acoustic extraction:** Single-pass extraction over all 5,604 train and 1,592 test clips producing `Rich-319d`, `HPSS-178d` (active `>-20 dB` + top-20% energy harmonic/percussive stats), `Mod_v1-468d` (10 ms hop), Std 40-mel patches (`w = 8, 32`) -> BoAW / VLAD / PCA-24 GMM-32 Fisher, and HPSS-Dual 24+24-mel patches (`w = 8, 32`) -> BoAW / VLAD / 2-seed bagged PCA-32 GMM-40 Fisher.
2. **Leakage-free 3-seed 5-fold CV & prior-debiased teacher:** Inside each fold (`SEEDS = [42, 913, 2026]`), a 6-view verified-only teacher (`0.8115` OOF plain MAP@3) is debiased by fold-internal uniform-weighted OOF marginals before top-1 unchecked agreement (`0.50` agree / `0.08` floor).
3. **Seven-stream classical ensemble + exact Debiased-Likelihood EM:** Combined `mkl` (`28%`), `jlr` (`18%`), `hpss_k` (`15%`), `std_k` (`10%`), `tab` (`12%`), `mod` (`10%`), and `knn` (`7%`). Retrained on 100% of verified + unchecked clips, divided out empirical 3-seed OOF marginals (`pi_src`), then applied temperature scaling (`T = 1.90`) and damped Saerens EM (`power = 0.50`).

### Changes from Submissions 8–10 (and team 0.85161 benchmark)
- Introduced Harmonic-Percussive Source Separation (`HPSS-178d` + HPSS-Dual 2-seed bagged Fisher/VLAD).
- Added Joint 8-View MKL SVM, convex L2 Multinomial Logistic Regression, and class-balanced Verified 16-NN.
- Replaced naive EM (`pi_src = 1/K`) with Cross-Fit OOF Marginal Prior-Debiasing (`+4.95%` to `+6.26%` in test-prior shift simulations).

### Validation results (3-Seed 5-Fold Stratified CV)
| Metric | Seed 42 | Seed 913 | Seed 2026 | **3-Seed Mean** |
| :--- | :---: | :---: | :---: | :---: |
| Plain OOF MAP@3 | `0.8766` | `0.8712` | `0.8688` | **`0.87222`** |
| Uniform-Weighted OOF MAP@3 | `0.8927` | `0.8870` | `0.8846` | **`0.88808`** |
| Train-Total Weighted OOF MAP@3 | — | — | — | **`0.90152`** |
| `v10` / `v40` OOF MAP@3 | — | — | — | **`0.89771` / `0.84127`** |

- Simulated test-prior shift (`DebiasEM 0.50`): Uniform `0.88464`, Train-Total `0.89798`, Skewed `0.88340`.